# 📊 Process data: A0–A3

Loại dòng thiếu `commit_date`, gộp dòng trùng, tạo feature, thống kê mô tả và kiểm tra chất lượng dữ liệu. `ai_tool` và `ai_verb` là heuristic, lấy từ mẫu tìm kiếm và comment trong `id`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
DATA_PATH = Path('../data/interim/latest_standardized_data.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Đã tải {DATA_PATH}: {len(df):,} dòng × {df.shape[1]} cột')

Đã tải ../data/interim/latest_standardized_data.csv: 35,278 dòng × 9 cột


## A0 — Chuẩn hóa kiểu ngày
Theo schema của nguồn, `change_commit_date` trống nghĩa là lần thay đổi đầu tiên không thu thập được, không phải code chưa từng bị sửa. Vì vậy `has_recorded_change` chỉ là cận dưới của việc code bị sửa.

In [2]:
for col in ['commit_date', 'change_commit_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
df['has_recorded_change'] = df['change_commit_date'].notna()
print(df[['commit_date', 'change_commit_date', 'change_commit_url']].isna().sum())

commit_date            1361
change_commit_date    22282
change_commit_url     22282
dtype: int64


## A1 — Lọc dòng, gộp dòng trùng và feature engineering
- Mỗi khối `(repo, file, code_block)` chỉ giữ một dòng. Một comment thường khớp với nhiều mẫu tìm kiếm nên cùng một khối xuất hiện nhiều lần. Khi các dòng trùng có `commit_date` khác nhau, giữ ngày sớm nhất.
- Loại tháng 4/2026 vì dữ liệu của tháng này chưa đủ.
- `commit_date` lấy từ `git blame` trên dòng comment ở một phiên bản file về sau, nên chỉ là cận trên của ngày code được đưa vào. Khoảng thời gian âm hoặc bằng 0 đến lần thay đổi đầu tiên là không hợp lệ và được chuyển thành NaN.

In [3]:
KEY = ['repo', 'file', 'code_block']
df_clean = df.dropna(subset=['commit_date'])
n_dated = len(df_clean)
# Trong một khối, các dòng có thay đổi có cùng change_commit_date: ưu tiên dòng có thay đổi, rồi commit_date sớm nhất
df_clean = df_clean.sort_values(['change_commit_date', 'commit_date'], na_position='last').drop_duplicates(KEY).sort_index()
n_unique = len(df_clean)
df_clean = df_clean[df_clean['commit_date'] < pd.Timestamp('2026-04-01', tz='UTC')].copy()
df_clean['year'] = df_clean['commit_date'].dt.year.astype('Int64')
df_clean['month'] = df_clean['commit_date'].dt.month.astype('Int64')
df_clean['year_month'] = df_clean['commit_date'].dt.strftime('%Y-%m')
df_clean['quarter'] = 'Q' + df_clean['commit_date'].dt.quarter.astype(str)
df_clean['hour'] = df_clean['commit_date'].dt.hour.astype('Int64')
df_clean['day_of_week'] = df_clean['commit_date'].dt.day_name()
df_clean['is_weekend'] = df_clean['commit_date'].dt.dayofweek >= 5
df_clean['days_to_modification'] = ((df_clean['change_commit_date'] - df_clean['commit_date']).dt.total_seconds() / 86400).round(3)
invalid_days = int((df_clean['days_to_modification'] <= 0).sum())
df_clean.loc[df_clean['days_to_modification'] <= 0, 'days_to_modification'] = np.nan
code = df_clean['code_block'].fillna('').astype(str)
# code_block lưu newline dưới dạng ký tự escape \\n; vẫn hỗ trợ newline thật nếu có
df_clean['code_lines'] = code.str.count(r'\\n').add(code.str.count('\n')).add(code.ne('').astype(int))
df_clean['code_chars'] = code.str.len()
# id = <language>__<mẫu tìm kiếm>__...::<dòng>::<dòng>::<comment>; không dùng code vì code chứa từ không liên quan (vd. DB cursor)
pattern = df_clean['id'].str.split('__').str[1].str.replace('_', ' ')
text = (pattern + ' ' + df_clean['id'].str.split('::', n=3).str[3].fillna('')).str.lower()
def first_label(series, rules, default='other_ai'):
    result = pd.Series(default, index=series.index, dtype='object')
    for label, pattern in rules:
        result.loc[result.eq(default) & series.str.contains(pattern, regex=True, na=False)] = label
    return result
# Các công cụ có rất ít khối được gộp vào other_tool để mô hình không bị tách biệt hoàn toàn (separation)
df_clean['ai_tool'] = first_label(text, [('chatgpt_gpt', r'chatgpt|gpt|openai'), ('github_copilot', r'copilot'), ('claude', r'claude|anthropic'), ('gemini_bard', r'gemini|bard'), ('llama_meta', r'llama|meta ai'), ('other_tool', r'deepseek|codewhisperer|tabnine|codeium|cursor (?:ai|ide|editor|agent)|(?:by|with|using|in|via) cursor\b')])
df_clean['ai_verb'] = np.where(pattern.str.contains('assist|suggest'), 'assisted', 'written')
df_clean['repo_commit_count'] = df_clean.groupby('repo')['commit_url'].transform('nunique')
print(f'Đã loại {len(df) - n_dated:,} dòng thiếu commit_date')
print(f'Đã gộp {n_dated - n_unique:,} dòng trùng → {n_unique:,} khối duy nhất')
print(f'Đã loại {n_unique - len(df_clean):,} khối của tháng 4/2026 → còn {len(df_clean):,} khối')
print(f'Có {invalid_days:,} khối có days_to_modification ≤ 0; đã chuyển thành NaN')
display(df_clean.head(3))

Đã loại 1,361 dòng thiếu commit_date
Đã gộp 12,415 dòng trùng → 21,502 khối duy nhất
Đã loại 4 khối của tháng 4/2026 → còn 21,498 khối
Có 1,742 khối có days_to_modification ≤ 0; đã chuyển thành NaN


,id,repo,language,commit_url,file,code_block,commit_date,change_commit_url,change_commit_date,has_recorded_change,year,month,year_month,quarter,hour,day_of_week,is_weekend,days_to_modification,code_lines,code_chars,ai_tool,ai_verb,repo_commit_count
0,python__written_by_gpt__433861003__e397ad21a6e...,flowerett/aoc,python,https://github.com/flowerett/aoc/commit/52a0fb...,2022/python/d1-gpt.py,# solution written by GPT chatbot\n# works and...,2022-12-03 08:41:16+00:00,NaN,NaT,False,2022,12,2022-12,Q4,8,Saturday,True,NaN,18,543,chatgpt_gpt,written,1
2,python__written_by_ai__437874902__14f89ee997ec...,tipa16384/adventofcode,python,https://github.com/tipa16384/adventofcode/comm...,2017/puzzle2.py,# Written by AI\n\ndef spreadsheet_checksum(li...,2022-12-04 05:58:44+00:00,NaN,NaT,False,2022,12,2022-12,Q4,5,Sunday,True,NaN,10,221,other_ai,written,1
4,python__written_by_ai__437874902__1d766ed4953e...,tipa16384/adventofcode,python,https://github.com/tipa16384/adventofcode/comm...,2017/puzzle1.py,"# Written by AI\n\nwith open('puzzle1.txt', 'r...",2022-12-04 05:58:44+00:00,NaN,NaT,False,2022,12,2022-12,Q4,5,Sunday,True,NaN,6,87,other_ai,written,1


## A2 — Descriptive statistics

In [4]:
numeric_cols = ['days_to_modification', 'code_lines', 'code_chars', 'repo_commit_count']
description = df_clean[numeric_cols].describe().T
description['skew'] = df_clean[numeric_cols].skew()
description['kurtosis'] = df_clean[numeric_cols].kurtosis()
display(description)
outlier_rows = []
for col in numeric_cols:
    s = df_clean[col].dropna(); q1, q3 = s.quantile([.25, .75]); iqr = q3-q1; lo, hi = q1-1.5*iqr, q3+1.5*iqr
    outlier_rows.append({'variable': col, 'q1': q1, 'q3': q3, 'lower_bound': lo, 'upper_bound': hi, 'outlier_count': int(((s < lo) | (s > hi)).sum())})
display(pd.DataFrame(outlier_rows))
for col in ['language', 'ai_tool', 'ai_verb', 'has_recorded_change']:
    print(f'\nTần suất: {col}')
    display(df_clean[col].value_counts(dropna=False).rename_axis(col).reset_index(name='count').head(20))
print('Crosstab language × has_recorded_change:')
display(pd.crosstab(df_clean['language'], df_clean['has_recorded_change'], margins=True))

,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
days_to_modification,5758.0,29.252684,71.594502,0.001,0.277,3.0865,21.18875,1011.911,5.143390,36.935664
code_lines,21498.0,97.891153,190.792527,2.000,12.000,32.0000,89.00000,2833.000,4.672161,30.965179
code_chars,21498.0,3721.488604,7546.538378,2.000,400.000,1146.5000,3291.75000,101285.000,4.759907,31.251563
repo_commit_count,21498.0,12.911899,37.297395,1.000,1.000,1.0000,2.00000,141.000,3.121261,7.816611


,variable,q1,q3,lower_bound,upper_bound,outlier_count
0,days_to_modification,0.277,21.18875,-31.090625,52.556375,873
1,code_lines,12.000,89.00000,-103.500000,204.500000,2699
2,code_chars,400.000,3291.75000,-3937.625000,7629.375000,2759
3,repo_commit_count,1.000,2.00000,-0.500000,3.500000,3920



Tần suất: language


,language,count
0,python,15781
1,js,5717



Tần suất: ai_tool


,ai_tool,count
0,other_ai,6600
1,chatgpt_gpt,6437
2,claude,4192
3,github_copilot,2702
4,gemini_bard,1113
5,llama_meta,419
6,other_tool,35


... (2 outputs left out)


Tần suất: has_recorded_change


,has_recorded_change,count
0,False,13998
1,True,7500


Crosstab language × has_recorded_change:


has_recorded_change,False,True,All
language,,,
js,3848,1869,5717
python,10150,5631,15781
All,13998,7500,21498


## A3 — Data quality checks
Khối có thay đổi nhưng khoảng thời gian ≤ 0 vẫn có `has_recorded_change=True` nhưng không có `days_to_modification`; outlier chỉ được báo cáo, không tự động xóa.

In [5]:
quality = {'rows': len(df_clean), 'columns': df_clean.shape[1], 'duplicate_blocks': int(df_clean.duplicated(KEY).sum()), 'duplicate_ids': int(df_clean['id'].duplicated().sum()), 'non_positive_days_original': invalid_days, 'missing_commit_date': int(df_clean['commit_date'].isna().sum()), 'commit_date_min': df_clean['commit_date'].min(), 'commit_date_max': df_clean['commit_date'].max()}
display(pd.Series(quality, name='value'))
assert df_clean['commit_date'].notna().all()
assert not df_clean.duplicated(KEY).any()
assert df_clean['has_recorded_change'].dtype == bool
assert (df_clean['days_to_modification'].dropna() > 0).all()
output_path = Path('../data/processed/commits_features.csv')
df_clean.to_csv(output_path, index=False)
print(f'✅ Đã xuất {output_path} ({output_path.stat().st_size / 1024 / 1024:.1f} MB)')
print(f'Shape: {df_clean.shape[0]:,} dòng × {df_clean.shape[1]} cột')

rows                                              21498
columns                                              23
duplicate_blocks                                      0
duplicate_ids                                         0
non_positive_days_original                         1742
missing_commit_date                                   0
commit_date_min               2022-12-03 08:41:16+00:00
commit_date_max               2026-03-31 17:15:58+00:00
Name: value, dtype: object

✅ Đã xuất ../data/processed/commits_features.csv (92.1 MB)
Shape: 21,498 dòng × 23 cột
